# Week 02 · Day 03 — LangGraph

**Topics:** StateGraph · Annotated reducers · Conditional routing · MemorySaver · interrupt_before


In [ ]:
%pip install -q langgraph langchain-openai

In [ ]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, api_key=os.getenv("OPENAI_API_KEY"))

## 1 · Minimal StateGraph

LangGraph models workflows as a directed graph. Nodes are Python functions; edges define the execution order. State is a typed dict that flows through the graph.

In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

# State definition
class SimpleState(TypedDict):
    input: str
    output: str

# Node functions — take state, return partial state update
def process_node(state: SimpleState) -> dict:
    result = llm.invoke(f"Summarize in one sentence: {state['input']}").content
    return {"output": result}

# Build graph
builder = StateGraph(SimpleState)
builder.add_node("process", process_node)
builder.add_edge(START, "process")
builder.add_edge("process", END)

graph = builder.compile()

result = graph.invoke({"input": "LangGraph is a library for building stateful, multi-actor applications with LLMs.", "output": ""})
print(result["output"])

## 2 · Annotated Reducers

By default, each node's return value **replaces** the state field. `Annotated[list, operator.add]` changes this: instead of replacing, the list is **appended to**. This lets multiple nodes accumulate results.

In [ ]:
class ResearchState(TypedDict):
    topic: str
    findings: Annotated[list[str], operator.add]  # accumulates — never overwrites
    summary: str
    steps: int

def researcher_1(state: ResearchState) -> dict:
    finding = llm.invoke(f"Give one key fact about: {state['topic']}").content
    return {"findings": [finding], "steps": 1}  # list gets appended

def researcher_2(state: ResearchState) -> dict:
    finding = llm.invoke(f"Give one surprising fact about: {state['topic']}").content
    return {"findings": [finding], "steps": 1}  # list gets appended

def synthesizer(state: ResearchState) -> dict:
    findings_text = "\n".join(f"- {f}" for f in state["findings"])
    summary = llm.invoke(f"Combine these findings into one paragraph:\n{findings_text}").content
    return {"summary": summary}

builder = StateGraph(ResearchState)
builder.add_node("researcher_1", researcher_1)
builder.add_node("researcher_2", researcher_2)
builder.add_node("synthesizer", synthesizer)

builder.add_edge(START, "researcher_1")
builder.add_edge(START, "researcher_2")  # parallel start
builder.add_edge("researcher_1", "synthesizer")
builder.add_edge("researcher_2", "synthesizer")
builder.add_edge("synthesizer", END)

graph = builder.compile()

result = graph.invoke({"topic": "quantum computing", "findings": [], "summary": "", "steps": 0})
print(f"Findings accumulated: {len(result['findings'])}")
for f in result["findings"]:
    print(f"  - {f[:80]}")
print(f"\nSummary: {result['summary'][:200]}")

## 3 · Conditional Routing

Use `add_conditional_edges` to route to different nodes based on state. Return the node name (or END) from the routing function.

In [ ]:
class QAState(TypedDict):
    question: str
    answer: str
    quality_score: float
    attempts: int

def answer_node(state: QAState) -> dict:
    answer = llm.invoke(state["question"]).content
    return {"answer": answer, "attempts": state["attempts"] + 1}

def evaluate_node(state: QAState) -> dict:
    import random
    # In production: use LLM-as-judge or RAGAS metrics
    score = random.uniform(0.5, 1.0)
    print(f"  [evaluate] attempt={state['attempts']}, score={score:.2f}")
    return {"quality_score": score}

def route_after_eval(state: QAState) -> str:
    if state["quality_score"] >= 0.80 or state["attempts"] >= 3:
        return END
    return "answer"  # retry

builder = StateGraph(QAState)
builder.add_node("answer", answer_node)
builder.add_node("evaluate", evaluate_node)

builder.add_edge(START, "answer")
builder.add_edge("answer", "evaluate")
builder.add_conditional_edges("evaluate", route_after_eval, {END: END, "answer": "answer"})

graph = builder.compile()

result = graph.invoke({"question": "Explain neural networks.", "answer": "", "quality_score": 0.0, "attempts": 0})
print(f"\nFinal answer (after {result['attempts']} attempt(s)):")
print(result["answer"][:200])

## 4 · MemorySaver — Persistent State Across Turns

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

class ChatState(TypedDict):
    messages: Annotated[list[dict], operator.add]

def chat_node(state: ChatState) -> dict:
    from langchain_core.messages import HumanMessage, AIMessage
    history = state["messages"]
    response = llm.invoke([HumanMessage(content=m["content"]) if m["role"] == "user" else AIMessage(content=m["content"]) for m in history])
    return {"messages": [{"role": "assistant", "content": response.content}]}

builder = StateGraph(ChatState)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# MemorySaver persists state between invocations for the same thread_id
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "conversation-1"}}

# Turn 1
r = graph.invoke({"messages": [{"role": "user", "content": "My favorite color is blue."}]}, config=config)
print(f"Turn 1: {r['messages'][-1]['content']}")

# Turn 2 — same thread_id, state carries over
r = graph.invoke({"messages": [{"role": "user", "content": "What's my favorite color?"}]}, config=config)
print(f"Turn 2: {r['messages'][-1]['content']}")

## 5 · Human-in-the-Loop with interrupt_before

In [ ]:
class ApprovalState(TypedDict):
    task: str
    plan: str
    result: str
    approved: bool

def plan_node(state: ApprovalState) -> dict:
    plan = llm.invoke(f"Create a 3-step plan to accomplish: {state['task']}").content
    return {"plan": plan}

def execute_node(state: ApprovalState) -> dict:
    result = f"Executed plan: {state['plan'][:50]}... [SUCCESS]"
    return {"result": result}

builder = StateGraph(ApprovalState)
builder.add_node("plan", plan_node)
builder.add_node("execute", execute_node)
builder.add_edge(START, "plan")
builder.add_edge("plan", "execute")
builder.add_edge("execute", END)

memory = MemorySaver()
# Pause BEFORE execute — human must approve the plan
graph = builder.compile(checkpointer=memory, interrupt_before=["execute"])

config = {"configurable": {"thread_id": "approval-001"}}
task = {"task": "Send a newsletter to 10,000 subscribers", "plan": "", "result": "", "approved": False}

# Run until interrupt
state_at_interrupt = graph.invoke(task, config=config)
print("Graph paused before 'execute' node.")
print(f"\nPlan for review:\n{state_at_interrupt['plan']}")

# Simulate human approval
human_approved = True  # in practice: API call, webhook, UI button

if human_approved:
    # Resume from interrupt — None means "continue from where we stopped"
    final = graph.invoke(None, config=config)
    print(f"\nResult: {final['result']}")
else:
    print("Plan rejected — not resuming.")

## Summary

- **StateGraph**: nodes = functions, edges = control flow, state = shared typed dict.
- **`Annotated[list, operator.add]`**: accumulates values instead of replacing them.
- **Conditional edges**: `add_conditional_edges(node, fn, mapping)` — `fn` returns the next node name.
- **MemorySaver**: persists state between `.invoke()` calls using `thread_id`.
- **`interrupt_before`**: pauses before a node; resume with `.invoke(None, config=config)`.

**Next:** [LLMOps](week02-day04-llmops.ipynb)